# JalanKita — Resume Training

Melanjutkan training dari checkpoint setelah session timeout.

**Prasyarat:**
- Input dataset `marcillleeeee/jalankita-checkpoint` sudah di-add
- Input dataset `marcillleeeee/jalankita-dataset-final` sudah di-add

| Kelas | Label |
|-------|-------|
| 0 | lubang_besar |
| 1 | lubang_kecil |
| 2 | retak_kulit_buaya |
| 3 | retak_memanjang |

---
## 1. Setup

In [ ]:
import os, sys, subprocess, json, shutil, glob, yaml, zipfile
import locale
locale.getpreferredencoding = lambda: 'UTF-8'

!pip install -q ultralytics

from ultralytics import YOLO
import torch
from pathlib import Path
from IPython.display import display, Image as DImage

print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')


---
## 2. Extract Dataset

In [ ]:
# Auto-discover dataset path — cari data.yaml di /kaggle/input/
import glob as _glob
yaml_candidates = _glob.glob('/kaggle/input/**/data.yaml', recursive=True)
if not yaml_candidates:
    raise FileNotFoundError('data.yaml tidak ditemukan di /kaggle/input/')
dataset_path = Path(yaml_candidates[0]).parent
DATASET_DIR = str(dataset_path)

print(f'Dataset path: {DATASET_DIR}')

# Copy data.yaml ke working dir dengan absolute path
with open(dataset_path / 'data.yaml') as f:
    data_cfg = yaml.safe_load(f)
data_cfg['path'] = DATASET_DIR
DATA_YAML = '/kaggle/working/data_fixed.yaml'
with open(DATA_YAML, 'w') as f:
    yaml.dump(data_cfg, f)
print(f'Fixed data.yaml -> {DATA_YAML}')

for split in ['train', 'valid', 'test']:
    img_dir = dataset_path / split / 'images'
    lbl_dir = dataset_path / split / 'labels'
    n_imgs = len(list(img_dir.glob('*'))) if img_dir.exists() else 0
    n_lbls = len(list(lbl_dir.glob('*'))) if lbl_dir.exists() else 0
    print(f'{split:6s}: {n_imgs:>6} images, {n_lbls:>6} labels')


---
## 3. Load Checkpoint

Mencari `last.pt` dari Input Dataset `marcillleeeee/jalankita-checkpoint`.
Checkpoint ini diupload otomatis tiap 25 epoch oleh `kaggle_train.ipynb`.

In [ ]:
# ── Cari checkpoint ──
CHECKPOINT_INPUT = '/kaggle/input/jalankita-checkpoint'
EXP_DIR = '/kaggle/working/jalankita_train/yolov8n_final'
WEIGHTS_DIR = f'{EXP_DIR}/weights'

# Cari di semua kemungkinan path (Kaggle mount bisa berbeda)
candidates = [
    os.path.join(CHECKPOINT_INPUT, 'last.pt'),
    os.path.join(CHECKPOINT_INPUT, 'jalankita-checkpoint', 'last.pt'),
]
candidates.extend(sorted(glob.glob(f'{CHECKPOINT_INPUT}/**/last.pt', recursive=True)))

last_pt = next((p for p in candidates if os.path.exists(p)), None)

if last_pt is None:
    raise FileNotFoundError(
        'last.pt tidak ditemukan di /kaggle/input/jalankita-checkpoint/.\n'
        'Pastikan dataset input sudah di-add ke notebook ini.'
    )

print(f'  Found: {last_pt}')

# Copy ke working directory persis struktur yang diharapkan YOLO
os.makedirs(WEIGHTS_DIR, exist_ok=True)
src_dir = os.path.dirname(last_pt)
for fname in ['last.pt', 'best.pt']:
    src = os.path.join(src_dir, fname)
    dst = os.path.join(WEIGHTS_DIR, fname)
    if os.path.exists(src):
        shutil.copy2(src, dst)
        print(f'  Restored: {dst}')

# Info checkpoint
ckpt = torch.load(last_pt, map_location='cpu', weights_only=False)
last_epoch = ckpt.get('epoch', -1) + 1
total_epochs = last_epoch
if 'train_args' in ckpt and isinstance(ckpt['train_args'], dict):
    total_epochs = int(ckpt['train_args'].get('epochs', last_epoch))
    data_path = ckpt['train_args'].get('data', '')
    print(f'  Data config: {data_path}')
remaining = total_epochs - last_epoch

print(f'  Checkpoint: epoch {last_epoch}/{total_epochs}')
print(f'  Remaining: ~{remaining} epochs')
print(f'  Mode: RESUME')


---
## 4. Resume Training

YOLO `resume=True` auto-membaca argumen dari checkpoint (optimizer, scheduler, augmentasi).
Auto-upload checkpoint tetap aktif tiap 25 epoch.

In [ ]:
CHECKPOINT_DATASET = 'marcillleeeee/jalankita-checkpoint'
CHECKPOINT_DIR = '/kaggle/working/checkpoint_upload'
CHECKPOINT_INTERVAL = 25

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
meta_path = os.path.join(CHECKPOINT_DIR, 'dataset-metadata.json')
if not os.path.exists(meta_path):
    with open(meta_path, 'w') as f:
        json.dump({
            "title": "JalanKita Checkpoint",
            "id": CHECKPOINT_DATASET,
            "licenses": [{"name": "CC0-1.0"}]
        }, f)

def upload_checkpoint(trainer):
    epoch = trainer.epoch + 1
    if epoch % CHECKPOINT_INTERVAL != 0 or epoch >= trainer.epochs:
        return
    weights_dir = trainer.save_dir / 'weights'
    for fname in ['last.pt', 'best.pt']:
        src = os.path.join(weights_dir, fname)
        if os.path.exists(src):
            shutil.copy2(src, os.path.join(CHECKPOINT_DIR, fname))
    try:
        result = subprocess.run(
            ['kaggle', 'datasets', 'version', '-p', CHECKPOINT_DIR,
             '-m', f'Checkpoint epoch {epoch}', '-q'],
            capture_output=True, text=True, timeout=120
        )
        if result.returncode != 0:
            result = subprocess.run(
                ['kaggle', 'datasets', 'create', '-p', CHECKPOINT_DIR, '-q'],
                capture_output=True, text=True, timeout=120
            )
        if result.returncode == 0:
            print(f'  \u2713 Checkpoint epoch {epoch} uploaded')
        else:
            print(f'  \u2717 Upload failed: {result.stderr.strip()[:200]}')
    except Exception as e:
        print(f'  \u2717 Upload error: {e}')


# ── Resume ──
model = YOLO(os.path.join(WEIGHTS_DIR, 'last.pt'))
model.add_callback('on_train_epoch_end', upload_checkpoint)

print(f'Resuming training from epoch {last_epoch}/{total_epochs}...')
results = model.train(
    resume=True,
    device='cuda' if torch.cuda.is_available() else 'cpu',
)

print('\nResume training selesai!')

BEST_PT = f'{EXP_DIR}/weights/best.pt'
BATCH_SIZE = 32
IMGSZ = 640


---
## 5. Evaluasi — Test Set

In [ ]:
if os.path.exists(BEST_PT):
    best_path = BEST_PT
elif os.path.exists(f'{EXP_DIR}/weights/last.pt'):
    best_path = f'{EXP_DIR}/weights/last.pt'
else:
    raise FileNotFoundError('No model found!')

best_model = YOLO(best_path)
val_results = best_model.val(
    data=DATA_YAML,
    split='test',
    imgsz=IMGSZ,
    batch=BATCH_SIZE,
    conf=0.25,
    iou=0.5,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    verbose=True
)

print('\n' + '='*60)
print('HASIL VALIDASI — TEST SET')
print('='*60)
print(f'mAP50-95:  {val_results.box.map:.4f}')
print(f'mAP50:     {val_results.box.map50:.4f}')
print(f'mAP75:     {val_results.box.map75:.4f}')
print(f'Precision: {val_results.box.mp:.4f}')
print(f'Recall:    {val_results.box.mr:.4f}')
f1 = 2 * val_results.box.mp * val_results.box.mr / (val_results.box.mp + val_results.box.mr + 1e-10)
print(f'F1-score:  {f1:.4f}')

print('\nPer-Class:')
for i, name in enumerate(best_model.names.values()):
    ap50 = val_results.box.ap50[i]
    ap5095 = val_results.box.ap[i].mean()
    p = val_results.box.p[i]
    r = val_results.box.r[i]
    f1 = 2 * p * r / (p + r + 1e-10)
    print(f'  {name:22s}: P={p:.4f} R={r:.4f} F1={f1:.4f} mAP50={ap50:.4f} mAP50-95={ap5095:.4f}')


---
## 6. Visualisasi

In [ ]:
cm = glob.glob(f'{EXP_DIR}/confusion_matrix*.png')
if not cm:
    cm = glob.glob(f'{EXP_DIR}/**/confusion_matrix*.png', recursive=True)
if cm:
    display(DImage(cm[-1]))
else:
    print('No confusion matrix found')


In [ ]:
preds = glob.glob(f'{EXP_DIR}/**/val_batch*_pred*', recursive=True)
if preds:
    display(DImage(preds[0]))
else:
    print('No validation sample found')


---
## 7. Export + Download

In [ ]:
best_model.export(format='onnx', imgsz=IMGSZ)
best_model.export(format='torchscript', imgsz=IMGSZ)
!ls -lh {EXP_DIR}/weights/


In [ ]:
output_zip = '/kaggle/working/jalankita_final_model.zip'
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(EXP_DIR):
        for f in files:
            if f.endswith(('.pt', '.onnx', '.torchscript', '.png', '.csv', '.yaml')):
                fp = os.path.join(root, f)
                arc = os.path.relpath(fp, EXP_DIR)
                zf.write(fp, arc)

print(f'Model package: {output_zip}')
print(f'Size: {os.path.getsize(output_zip) / 1e6:.1f} MB')
